# Colorado — CRS Title 10 (Insurance) → `data/colorado/ins_codes/*.md`

Colorado insurance statutes are **Title 10 — Insurance** of the **Colorado Revised Statutes (C.R.S.)**.

The Office of Legislative Legal Services publishes each title as a **single HTML file** on **olls.info**, for example **`crs2024-title-10.htm`**. That file is plain HTML (no JavaScript paywall): one **`httpx`** download, then **BeautifulSoup** walks `<p>` blocks whose first `<strong>` is a section caption (`10-x-yyy.` …).

This notebook:

1. Downloads **`https://olls.info/crs/crs{YEAR}-title-10.htm`** (see **`CRS_YEAR`**).
2. Merges consecutive duplicate section numbers (the source sometimes prints two versions, e.g. renumbered / editor’s note).
3. Writes one **`COL_sec_10_x_yyy.md`** per section (dots in numbers become underscores).

**Official hub:** [Colorado Revised Statutes — General Assembly](https://content.leg.colorado.gov/agencies/office-legislative-legal-services/colorado-revised-statutes) (links to OLLS title downloads).

**Politeness:** one large request by default; **`REQUEST_DELAY_SEC`** applies before the download (set higher if you re-run often).

Then run **`python -m app.ingest`** from the project root.


In [1]:
%pip install -q httpx beautifulsoup4


You should consider upgrading via the '/Users/apps/Downloads/ZProjects/RAG/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import re
import time
from pathlib import Path

import httpx
from bs4 import BeautifulSoup

# Calendar year on the OLLS file name (e.g. 2024 → crs2024-title-10.htm).
CRS_YEAR = "2024"
TITLE_NUM = 10
SOURCE_URL = f"https://olls.info/crs/crs{CRS_YEAR}-title-{TITLE_NUM}.htm"

OUT_DIR = Path("data") / "colorado" / "ins_codes"
OUT_DIR.mkdir(parents=True, exist_ok=True)

USER_AGENT = "RAG-CO-CRS-10/1.0 (public Colorado Revised Statutes; educational indexing)"
REQUEST_DELAY_SEC = 0.35
TIMEOUT = 180.0

# 0 = export every section found in the title file
MAX_SECTIONS = 0

SKIP_EXISTING = True

SECTION_STRONG = re.compile(r"^\s*10-\d+-\d+(?:\.\d+)?\.\s+.+", re.I)
SECTION_ID = re.compile(r"^\s*(10-\d+-\d+(?:\.\d+)?)\.", re.I)


In [3]:
def starts_section_id(p):
    st = p.find("strong")
    if not st:
        return None
    t = st.get_text()
    if not SECTION_STRONG.match(t):
        return None
    m = SECTION_ID.match(t)
    return m.group(1) if m else None


def collect_section_text(start_p) -> str:
    parts = [start_p.get_text("\n", strip=True)]
    for sib in start_p.find_next_siblings("p"):
        if starts_section_id(sib):
            break
        parts.append(sib.get_text("\n", strip=True))
    return "\n\n".join(x for x in parts if x)


def sid_to_filename(sid: str) -> str:
    safe = sid.replace("-", "_").replace(".", "_")
    return f"COL_sec_{safe}.md"


def parse_title10_html(html: str) -> list[tuple[str, str]]:
    """Return [(section_id, plain_text), ...] in document order."""
    soup = BeautifulSoup(html, "html.parser")
    body = soup.find("body") or soup
    section_ps: list[tuple[str, object]] = []
    for p in body.find_all("p"):
        sid = starts_section_id(p)
        if sid:
            section_ps.append((sid, p))

    merged: list[tuple[str, str]] = []
    i = 0
    while i < len(section_ps):
        sid, p = section_ps[i]
        chunks = [collect_section_text(p)]
        while i + 1 < len(section_ps) and section_ps[i + 1][0] == sid:
            i += 1
            chunks.append(collect_section_text(section_ps[i][1]))
        text = chunks[0] if len(chunks) == 1 else "\n\n---\n\n".join(chunks)
        merged.append((sid, text))
        i += 1
    return merged


def download_title10() -> dict[str, int]:
    time.sleep(REQUEST_DELAY_SEC)
    with httpx.Client(
        headers={"User-Agent": USER_AGENT, "Accept": "text/html,*/*;q=0.8"},
        timeout=TIMEOUT,
        follow_redirects=True,
        http2=False,
    ) as client:
        r = client.get(SOURCE_URL)
        r.raise_for_status()
        html = r.text

    (OUT_DIR / "_colorado_crs_title10_source.txt").write_text(SOURCE_URL + "\n", encoding="utf-8")
    pairs = parse_title10_html(html)
    print(f"Parsed {len(pairs)} sections from {SOURCE_URL}")

    todo = pairs if not MAX_SECTIONS else pairs[:MAX_SECTIONS]
    if MAX_SECTIONS:
        print(f"Limited export to first {len(todo)} sections (MAX_SECTIONS)")

    wrote, skipped = 0, 0
    for sid, body in todo:
        dest = OUT_DIR / sid_to_filename(sid)
        if SKIP_EXISTING and dest.exists() and dest.stat().st_size > 80:
            skipped += 1
            continue
        title = f"Colorado Revised Statutes — Title 10 (Insurance) § {sid}"
        md = (
            f"# {title}\n\n"
            f"**Colorado Revised Statutes — Title {TITLE_NUM} (Insurance)**\n\n"
            f"**Official source (OLLS HTML):** {SOURCE_URL}\n\n"
            f"**Section:** {sid}\n\n"
            f"---\n\n"
            f"{body}\n"
        )
        dest.write_text(md, encoding="utf-8")
        wrote += 1

    print(f"Done. wrote={wrote} skipped={skipped} → {OUT_DIR.resolve()}")
    return {"wrote": wrote, "skipped": skipped}


download_title10()


Parsed 1341 sections from https://olls.info/crs/crs2024-title-10.htm
Done. wrote=1341 skipped=0 → /Users/apps/Downloads/ZProjects/RAG/ins_ipynb/data/colorado/ins_codes


{'wrote': 1341, 'skipped': 0}

## Next step

`python -m app.ingest` from the repository root.
